# 03 · Evaluate & analyze errors
Loads a trained checkpoint (or falls back to the rule baseline), scores it on every split, and drills into the error taxonomy from `src/evaluation/error_analysis.py`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
from src.datasets.pipeline import load_split
from src.evaluation.evaluate import evaluate_rows
from src.evaluation.error_analysis import analyze, build_train_vocab
from src.inference.generate import translate_texts
from src.models.factory import load_checkpoint
from src.models.rule_baseline import RuleBaseline
import pandas as pd
PROCESSED = '../data/processed'

In [ ]:
import pathlib as _p
ckpt = _p.Path('../experiments/checkpoints/attn_bahdanau/best.pt')
use_neural = ckpt.exists()
if use_neural:
    model, src_tok, tgt_tok, meta = load_checkpoint(ckpt)
    def predict(texts):
        return [h[0]['text'] for h in translate_texts(model, src_tok, tgt_tok, texts, beam_size=4)]
    print('Using neural checkpoint:', ckpt)
else:
    rb = RuleBaseline.load(f'{PROCESSED}/rule_baseline.json')
    predict = rb.normalize_many
    print('No neural checkpoint found yet -- using the rule baseline. Train one with scripts/train.py.')

In [ ]:
rows = load_split(PROCESSED, 'test')
preds = predict([r['src'] for r in rows])
result = evaluate_rows(rows, preds)
result['overall']

## Sliced metrics (by mix ratio, script, noise kind, length)

In [ ]:
for slice_name, groups in result['slices'].items():
    print(f'--- {slice_name} ---')
    print(pd.DataFrame(groups).T[['n','em','cer','chrf']])


## Error taxonomy

In [ ]:
train_vocab = build_train_vocab(load_split(PROCESSED, 'train'))
err = analyze(rows, preds, train_vocab)
print('sentence error rate:', round(err['sentence_error_rate'], 3))
pd.Series(err['categories']).sort_values(ascending=False)

In [ ]:
# A few concrete examples per category
for cat, examples in list(err['examples'].items())[:5]:
    print(f'== {cat} ==')
    for ex in examples[:2]:
        print(f"  src={ex['src']!r}\n  ref={ex['ref']!r}\n  hyp={ex['hyp']!r}")
    print()